In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import PSD_utils
from scipy.signal import welch

In [ ]:
plt.rcParams.update({
    "text.usetex": False,  
    "mathtext.fontset": "cm", 
    "font.family": "serif",
    "font.serif": ["Latin Modern Roman", "Computer Modern Roman", "DejaVu Serif"],
    
    "xtick.labelsize": 18,  
    "ytick.labelsize": 18, 
})

In [ ]:
power = '0p30'

base = f"/disk/hyk049/DHM_new_1Dcenter/{power}/"

def h5read_T(fname, dset):
    with h5py.File(fname, "r") as f:
        return np.array(f[dset])

t = h5read_T(base + f"Q_1D_{power}vpp_b.h5", "/t")
x = h5read_T(base + f"Q_1D_{power}vpp_b.h5", "/x")
nx = len(x)

labels = list("abcdefghijklmnop") 

# Load all Q matrices
Q = {}
for lbl in labels:
    fname = f"{base}Q_1D_{power}vpp_{lbl}.h5"
    Q[lbl] = h5read_T(fname, "/Q_1D")
    

In [7]:
dt = t[1] - t[0]
t_dep = 0.05
seg_len = int(t_dep//dt)+1
num_segs = len(t)//seg_len

Q_a = Q["b"]
nx, nt = Q_a.shape
t_new = seg_len * num_segs

Q_a_split = np.split(Q_a[:,:t_new], num_segs, axis=1)

In [ ]:
FS_DHM = 115_200
min_pts = 10
target_segments_full = 100

i_center = 100  # PSD at the center point

t_deps = [0.01, 0.05, 0.1, 0.15]
target_segs = [5, 10, 10, 10]

labels  = list(Q.keys())

seg_lens_list = [int(t_dep / dt) + 1 for t_dep in t_deps]
num_segs_list = [len(t) // seg_len for seg_len in seg_lens_list]

ALL_psds         = {t_dep: {} for t_dep in t_deps}
ALL_psds_alltime = {t_dep: {} for t_dep in t_deps}

#  COMPUTE PSDs
for t_dep, seg_len, num_segs, target_seg in zip(t_deps, seg_lens_list, num_segs_list, target_segs):
    print(rf"Computing PSDs for t_d={t_dep}s...")
    psd_list = []
    
    for lbl in labels:
        Q_mat = Q[lbl]

        t_new = seg_len * num_segs
        Q_split = np.split(Q_mat[:, :t_new], num_segs, axis=1)    

        for seg in Q_split:
            k_seg, psd_seg, f_seg, *_ = PSD_utils.compute_PSD(seg[i_center, :], FS_DHM, min_pts, target_seg)
            psd_list.append((k_seg, psd_seg, f_seg))

        ALL_psds[t_dep] = psd_list
        
    # PSD for the entire time series (for comparison)
    k_seg, psd_alltime, f_seg_alltime, *_ = PSD_utils.compute_PSD(Q_mat[i_center, :], FS_DHM, min_pts, 100)
    ALL_psds_alltime[t_dep] = psd_alltime

Computing PSDs for t_d=0.01s...
Computing PSDs for t_d=0.05s...


In [ ]:
# Compute stats for PSDs
PSD_stats = {}

for t_dep in t_deps:
    PSD_stats[t_dep] = {}

    psds = []
    for i in range(len(ALL_psds[t_dep])):
        _, psd, _ = ALL_psds[t_dep][i]
        psds.append(psd)
        
    psds = np.array(psds)
    mean_psd = np.mean(psds, axis=0)
    var_psd  = np.var(psds, axis=0)

    PSD_stats[t_dep] = {
        "mean": mean_psd,
        "var": var_psd,
    }

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True, sharey=True)
axes = axes.flatten()

for i, t_dep in enumerate(t_deps):
    ax = axes[i]

    first = True
        
    for k_seg, psd_seg, f_seg in ALL_psds[t_dep]:
        L = len(ALL_psds[t_dep])
        ax.plot(f_seg, psd_seg, lw=0.5, alpha=0.05, color='C1')
        
        first = False
    
    # PSD stats
    stats = PSD_stats[t_dep]
    mean = stats["mean"]
    var = stats["var"]
    std = np.sqrt(var)
    var_mean = np.mean(var) # average variance
    
    ax.plot(f_seg, mean, color='C3', lw=2, label=f"mean w/ var=${var_mean:.3f}$")
    
    # All time frame psd
    ALL_psds_alltime_temp = ALL_psds_alltime[t_dep]
    ax.plot(f_seg_alltime, ALL_psds_alltime_temp, lw=1.5, ls='--', color='C0', label=r"full time frame ($T_f$)")
        
    # ax.set_xscale('log', base=2)
    # ax.set_xlim(log2_xticks.min()/1.3, log2_xticks.max()*1.3)
    
    ax.set_xscale('log')
    ax.set_xlabel('Frequency [Hz]', fontsize=20)
    ax.tick_params(axis='both', labelsize=20)
    ax.set_title(rf"$t_d = {t_dep}$s ($L={L}$)", fontsize=20)
    ax.grid(True, which='both', ls='--', alpha=0.5)
    ax.legend(prop={'family': 'serif', 'size': 16})
    ax.set_xlim([300, 60000])

fig.text(0.04, 0.5, r"$\log_{10}(\mathrm{PSD})$ [$\mu$m$^2$/(Hz)]", va='center', rotation='vertical', fontsize=20)

plt.tight_layout(rect=[0.05, 0.05, 1, 0.98])
plt.show()